# Article 10 — Statistical Exploration of the Manifold Hypothesis

**Paper:** *Statistical exploration of the Manifold Hypothesis*, Nick Whiteley, Annie Gray, Patrick Rubin-Delanchy (2025). Read before the Royal Statistical Society, October 2025.

This notebook reproduces the **torus example from Section 3.5**. The authors propose the **Latent Metric Space (LMS) model**: a statistical model that produces manifold structure in high-dimensional data from latent variables, correlation, and stationarity. We verify three key results: data inner products reflect manifold inner products (Proposition 1), the manifold $M$ is homeomorphic to $Z$ (Proposition 2), and under stationarity $M$ is isometric to $Z$ up to scaling (Proposition 3).

In [ ]:
%matplotlib inline

import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
from src.torus import sample_torus
from src.lms_model import generate_lms_data
from src.pca_embedding import pca_embed, spherical_project
from src.graph_analysis import build_knn_graph, compute_shortest_paths, sample_pairs
from src.plotting import plot_torus, plot_pca_dims, plot_norms_kde, plot_isometry, plot_varying_p

## Step 1 — Latent space $Z$ is a torus (Section 3.5)

The latent variables $z_1, \dots, z_n$ are sampled uniformly on a torus $\mathbb{T}^2 \subset \mathbb{R}^3$ with major radius $R=2$ and minor radius $r=1$. This is the running example in Section 3.5, illustrated in Figure 3 of the paper.

In [ ]:
Z, theta1, theta2 = sample_torus(n=1500)
print(f"Torus points: {Z.shape}")
fig = plot_torus(Z, theta1, theta2)

## Step 2 — LMS model data generation (Section 2, Eq. 1)

The LMS model generates high-dimensional observations as $Y_{ij} = X_j(z_i) + \sigma \varepsilon_{ij}$, where each $X_j$ is drawn from a Gaussian process with kernel $f(z, z') = \exp(-\|z - z'\|^2)$ and $\varepsilon_{ij} \sim \mathcal{N}(0,1)$ is independent noise. We use $p = 2000$ features and noise level $\sigma = 0.5$.

In [ ]:
Y, X = generate_lms_data(Z, p=2000, sigma=0.5)
print(f"Observed data Y: {Y.shape}  (n={Y.shape[0]}, p={Y.shape[1]})")
print(f"Signal-to-noise ratio p/n = {Y.shape[1]/Y.shape[0]:.1f}")

## Step 3 — PCA embedding recovers the feature map (Theorem 1)

Theorem 1 shows that the rescaled PCA scores $\zeta_i = p^{-1/2} \hat{U}_r \hat{\Lambda}_r$ converge to the true feature map $\mu(z_i)$ as $p \to \infty$. We compute a 9-dimensional embedding to visualize dimensions 1–3, 4–6, and 7–9, reproducing Figure 4 of the paper.

In [ ]:
zeta_9 = pca_embed(Y, r=9)
print(f"PCA embedding (r=9): {zeta_9.shape}")
fig = plot_pca_dims(zeta_9, theta1, theta2)

## Step 4 — Spherical projection (Section 4.3)

The PCA embedding $\zeta_i$ approximates the feature map up to a rotation, but the norms $\|\zeta_i\|$ vary. Section 4.3 projects onto the unit sphere: $\tilde{\zeta}_i = \zeta_i / \|\zeta_i\|$. The KDE below shows the norm variation that this projection removes.

In [ ]:
zeta_15 = pca_embed(Y, r=15)
zeta_sp, norms = spherical_project(zeta_15)
print(f"Spherical embedding: {zeta_sp.shape}")
print(f"Norm range: [{norms.min():.3f}, {norms.max():.3f}]")
fig = plot_norms_kde(norms)

## Step 5 — Isometry verification (Proposition 3, Figure 5)

Proposition 3 states that under stationarity, the manifold $M$ is isometric to $Z$ up to a scaling factor $\sqrt{-2g'(0)}$. For the Gaussian kernel $f(z,z') = e^{-\|z-z'\|^2}$, this gives a scaling of $\sqrt{2}$. We verify this by comparing graph geodesics on the torus $Z$ and on the spherical PCA embedding $\hat{M}$.

In [ ]:
pairs = sample_pairs(len(Z), n_pairs=300)

graph_Z = build_knn_graph(Z, k=8, metric='euclidean')
graph_M = build_knn_graph(zeta_sp, k=8, metric='cosine')

dist_Z = compute_shortest_paths(graph_Z, pairs)
dist_M = compute_shortest_paths(graph_M, pairs)

scaling_factor = np.sqrt(2)
fig = plot_isometry(dist_Z, dist_M, scaling_factor)

## Step 6 — Effect of $p$ on manifold recovery (Theorem 1)

Theorem 1 guarantees that the PCA embedding converges to the feature map as $p \to \infty$. We fix $n = 1000$ and $\sigma = 0.5$, and vary $p \in \{200, 1000, 5000\}$ to visualize this convergence: the torus structure becomes progressively cleaner with increasing $p$.

In [ ]:
Z_exp, theta1_exp, _ = sample_torus(n=1000, seed=42)

results = {}
for p in [200, 1000, 5000]:
    Y_exp, _ = generate_lms_data(Z_exp, p=p, sigma=0.5, seed=42)
    results[p] = pca_embed(Y_exp, r=3)
    print(f"p={p}: done")

fig = plot_varying_p(results, theta1_exp)

## Conclusion

We reproduced the torus example from Section 3.5 of Whiteley, Gray & Rubin-Delanchy (2025). The PCA embedding of LMS-generated data recovers the torus topology (Proposition 2), the spherical projection followed by graph geodesic comparison confirms the isometry up to scaling by $\sqrt{2}$ (Proposition 3), and increasing $p$ improves the quality of the recovered manifold (Theorem 1).